# INES Tutorial: Probe Panel Design for Spatial Transcriptomics

This tutorial demonstrates how to use INES (Imputation Noise Evaluation Score) to design optimized gene panels for spatial transcriptomics experiments like MERFISH and Xenium.

## Overview

In this tutorial, you will learn how to:
1. Load and prepare scRNA-seq data
2. Calculate INES scores to identify unimputable genes
3. Design an optimized probe panel
4. Compare INES with traditional methods (HVG, co-expression)
5. Evaluate panel quality

## Setup

First, let's import the necessary libraries:

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

# Import INES functions
from ines.metrics import calculate_ines_score, compute_gene_reliability_matrix
from ines.design import optimize_gene_panel, greedy_probe_selection
from ines.models import ImputationSimulator

# Set plotting style
sns.set_style("whitegrid")
sc.settings.verbosity = 3

## Step 1: Load and Prepare Data

For this tutorial, we'll use a PBMC dataset. Replace this with your own data.

In [ ]:
# Load example data (replace with your dataset)
# adata = sc.read_h5ad('your_data.h5ad')

# For demonstration, we'll load a public dataset
adata = sc.datasets.pbmc3k()

# Basic preprocessing
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

print(f"Dataset: {adata.n_obs} cells x {adata.n_vars} genes")

## Step 2: Simulate Imputation

For this tutorial, we'll simulate imputation using the INES simulator. In practice, you would use your own imputation method (MAGIC, DCA, scVI, etc.) and store the results in `adata.layers['imputed']`.

In [ ]:
# Normalize and log-transform
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Store raw counts
adata.layers['raw'] = adata.X.copy()

# Simulate imputation
simulator = ImputationSimulator(model_type='knn', random_state=42)

# Convert to dense array if sparse
if hasattr(adata.X, 'toarray'):
    data_array = adata.X.toarray()
else:
    data_array = adata.X

# Perform imputation
imputed_data = simulator.simulate_imputation(
    data_array,
    dropout_rate=0.1,
    noise_level=0.05
)

# Store imputed data
adata.layers['imputed'] = imputed_data

print("Imputation completed!")

## Step 3: Calculate INES Scores

Now we'll calculate INES scores for all genes. Higher scores indicate genes that are poorly imputed and should be included in the spatial panel.

In [ ]:
# Calculate INES scores
ines_scores, gene_names = compute_gene_reliability_matrix(
    adata,
    imputed_layer='imputed',
    raw_layer='raw',
    n_top_genes=2000  # Focus on highly variable genes
)

# Create a dataframe for easy analysis
ines_df = pd.DataFrame({
    'gene': gene_names,
    'ines_score': ines_scores
}).sort_values('ines_score', ascending=False)

print(f"\nTop 10 genes with highest INES scores (most unimputable):")
print(ines_df.head(10))

print(f"\nBottom 10 genes with lowest INES scores (most imputable):")
print(ines_df.tail(10))

## Step 4: Visualize INES Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(ines_scores, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('INES Score')
axes[0].set_ylabel('Number of Genes')
axes[0].set_title('Distribution of INES Scores')
axes[0].axvline(np.median(ines_scores), color='red', linestyle='--', label='Median')
axes[0].legend()

# Ranked plot
axes[1].plot(range(len(ines_scores)), sorted(ines_scores, reverse=True))
axes[1].set_xlabel('Gene Rank')
axes[1].set_ylabel('INES Score')
axes[1].set_title('Ranked INES Scores')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Design Probe Panel

Now let's use INES to design an optimized probe panel. We'll compare INES-based selection with traditional methods.

In [ ]:
# Number of probes to select (typical for spatial experiments)
n_probes = 300

# INES-based selection
ines_panel = optimize_gene_panel(
    adata,
    n_probes=n_probes,
    imputed_layer='imputed',
    strategy='ines',
    correlation_threshold=0.3
)

print(f"\nINES Panel: Selected {len(ines_panel['genes'])} genes")
print(f"Top 20 selected genes: {ines_panel['genes'][:20]}")
print(f"Mean INES score: {np.mean(ines_panel['ines_scores']):.3f}")

## Step 6: Compare with Traditional Methods

Let's compare INES with HVG (Highly Variable Genes) selection.

In [ ]:
# HVG-based selection
hvg_panel = optimize_gene_panel(
    adata,
    n_probes=n_probes,
    strategy='hvg'
)

# Combined strategy
combined_panel = optimize_gene_panel(
    adata,
    n_probes=n_probes,
    imputed_layer='imputed',
    strategy='combined'
)

print(f"\nHVG Panel: {len(hvg_panel['genes'])} genes")
print(f"Combined Panel: {len(combined_panel['genes'])} genes")

# Compare overlap
ines_set = set(ines_panel['genes'])
hvg_set = set(hvg_panel['genes'])
overlap = ines_set & hvg_set

print(f"\nOverlap between INES and HVG: {len(overlap)} genes ({100*len(overlap)/n_probes:.1f}%)")
print(f"INES-unique genes: {len(ines_set - hvg_set)}")
print(f"HVG-unique genes: {len(hvg_set - ines_set)}")

## Step 7: Visualize Panel Comparison

In [ ]:
# Create comparison dataframe
comparison_data = []
for gene in ines_df['gene']:
    in_ines = gene in ines_set
    in_hvg = gene in hvg_set
    score = ines_df[ines_df['gene'] == gene]['ines_score'].values[0]
    
    if in_ines and in_hvg:
        category = 'Both'
    elif in_ines:
        category = 'INES only'
    elif in_hvg:
        category = 'HVG only'
    else:
        category = 'Neither'
    
    comparison_data.append({
        'gene': gene,
        'ines_score': score,
        'category': category
    })

comparison_df = pd.DataFrame(comparison_data)

# Plot
plt.figure(figsize=(10, 6))
for category in ['INES only', 'HVG only', 'Both']:
    subset = comparison_df[comparison_df['category'] == category]
    plt.scatter(
        range(len(subset)),
        subset['ines_score'].values,
        label=category,
        alpha=0.6,
        s=20
    )

plt.xlabel('Gene Index')
plt.ylabel('INES Score')
plt.title('Gene Selection Comparison: INES vs HVG')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Step 8: Save Results

Finally, let's save the selected gene panel for use in spatial transcriptomics experiments.

In [ ]:
# Save INES panel
ines_panel_df = pd.DataFrame({
    'gene': ines_panel['genes'],
    'ines_score': ines_panel['ines_scores']
})
ines_panel_df.to_csv('ines_probe_panel.csv', index=False)

# Save all INES scores
ines_df.to_csv('all_ines_scores.csv', index=False)

print("Results saved!")
print("- ines_probe_panel.csv: Selected gene panel")
print("- all_ines_scores.csv: INES scores for all genes")

## Conclusion

In this tutorial, you learned how to:
- Calculate INES scores to identify genes that are difficult to impute
- Design optimized probe panels for spatial transcriptomics
- Compare INES-based selection with traditional methods
- Export results for downstream analysis

### Next Steps

1. Try INES with your own dataset and imputation method
2. Experiment with different `n_probes` and `correlation_threshold` values
3. Validate the panel using the benchmarking scripts
4. Use the selected genes in your spatial transcriptomics experiment!

For more information, see the [INES documentation](https://github.com/Pixel-Dream/ines_dev).